In [1]:
#Loading in Packages and Data

#Importing Packages
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.ticker as ticker
import matplotlib.cm as cm
from matplotlib.colors import Normalize
from matplotlib.ticker import MaxNLocator
from matplotlib.ticker import ScalarFormatter
import matplotlib.gridspec as gridspec
import xarray as xr
import os; import time
import pickle
import h5py
###############################################################
def coefs(coefficients,degree):
    coef=coefficients
    coefs=""
    for n in range(degree, -1, -1):
        string=f"({coefficients[len(coef)-(n+1)]:.1e})"
        coefs+=string + f"x^{n}"
        if n != 0:
            coefs+=" + "
    return coefs
###############################################################

# #Importing Model Data
    
# dir='/mnt/lustre/koa/koastore/torri_group/air_directory/DCI-Project/'
# data=xr.open_dataset(dir+'../cm1r20.3/run/cm1out_test7tundra-7_062217.nc') #***
# true_time=data['time']
# # parcel=xr.open_dataset(dir+'../cm1r20.3/run/cm1out_pdata_test5tundra-7_062217.nc') #***
# times=data['time'].values/(1e9 * 60); times=times.astype(float);
# Np_str='125e3'
# #Restricts the timesteps of the data from timesteps0 to 140
# data=data.isel(time=np.arange(0,140+1))
# # parcel=parcel.isel(time=np.arange(0,140+1))
# res='1km'

dir='/mnt/lustre/koa/koastore/torri_group/air_directory/DCI-Project/'
data=xr.open_dataset(dir+'../cm1r20.3/run/cm1out_1km_1e6.nc') #***
true_time=data['time']
parcel=xr.open_dataset(dir+'../cm1r20.3/run/cm1out_pdata_1km_1e6.nc') #***
times=data['time'].values/(1e9 * 60); times=times.astype(float);
Np_str='1e6'
#Restricts the timesteps of the data from timesteps0 to 140
res='1km'
index_adjust=0
ocean_fraction=0.25


# #uncomment if using 250m data
# #Importing Model Data
# check=False
# dir2='/home/air673/koa_scratch/'
# data=xr.open_dataset(dir2+'cm1out_250m.nc') #***
# # # parcel=xr.open_dataset(dir2+'cm1out_pdata_250m.nc') #***

# # Restricts the timesteps of the data from timesteps0 to 140
# data=data.isel(time=np.arange(0,400+1))
# # # parcel=parcel.isel(time=np.arange(0,400+1))
# res='250m'

In [2]:
import sys
dir2='/mnt/lustre/koa/koastore/torri_group/air_directory/DCI-Project/'
path=dir2+'../Functions/'
sys.path.append(path)

import NumericalFunctions
from NumericalFunctions import * # import NumericalFunctions 
import PlottingFunctions
from PlottingFunctions import * # import PlottingFunctions


# # Get all functions in NumericalFunctions
# import inspect
# functions = [f[0] for f in inspect.getmembers(NumericalFunctions, inspect.isfunction)]
# functions

# # Get all functions in NumericalFunctions
# import inspect
# functions = [f[0] for f in inspect.getmembers(PlottingFunctions, inspect.isfunction)]
# functions

In [15]:
# Reading Back Data Later
##############
import h5py
dir2=dir+'Project_Algorithms/Lagrangian_Binary_Array/'
in_file=dir2+f'lagrangian_binary_array_{res}_{Np_str}.h5'
# in_file=dir2+f'lagrangian_binary_array_{res}_{Np_str}_TEST.h5'
with h5py.File(in_file, 'r') as f:
    # Load the dataset by its name
    A_g = f['A_g'][:]
    A_c = f['A_c'][:]

    # W = f['W'][:]
    # QCQI = f['QCQI'][:]
    Z = f['Z'][:]
    Y = f['Y'][:]
    X = f['X'][:]

# #Making Time Matrix
# rows, cols = A.shape[0], A.shape[1]
# T = np.arange(rows).reshape(-1, 1) * np.ones((1, cols), dtype=int)

In [4]:
#READING BACK IN
mins_thresh=5
# mins_thresh=10
dir3=dir+f'Project_Algorithms/Entrainment/processed_binary_arrays_'+str(mins_thresh)+f'mins_{res}_{Np_str}.h5'
with h5py.File(dir3, 'r') as h5file:
    A_g_Processed = h5file['A_g_Processed'][:]
    A_c_Processed = h5file['A_c_Processed'][:]

In [5]:
#job array setup
job_array=True
if job_array==True:

    num_jobs=200 #how many total jobs are being run? i.e. array=1-100 ==> num_jobs=100 #***
    total_elements=len(parcel['xh']) #total num of variables

    if num_jobs >= total_elements:
        raise ValueError("Number of jobs cannot be greater than or equal to total elements.")
    
    job_range = total_elements // num_jobs  # Base size for each chunk
    remaining = total_elements % num_jobs   # Number of chunks with 1 extra 
    
    # Function to compute the start and end for each job_id
    def get_job_range(job_id):
        job_id-=1
        # Add one extra element to the first 'remaining' chunks
        start_job = job_id * job_range + min(job_id, remaining)
        end_job = start_job + job_range + (1 if job_id < remaining else 0)
    
        if job_id == num_jobs - 1: 
            end_job = total_elements - 1
        return start_job, end_job
    def job_testing():
        #TESTING
        start=[];end=[]
        for job_id in range(1,num_jobs+1):
            start_job, end_job = get_job_range(job_id)
            print(start_job,end_job)
            start.append(start_job)
            end.append(end_job)
        print(np.all(start!=end))
        print(len(np.unique(start))==len(start))
        print(len(np.unique(end))==len(end))
    # job_testing()
    
    job_id = int(os.environ.get('SLURM_ARRAY_TASK_ID', 0)) #this is the current SBATCH job id
    if job_id==0: job_id=1
    start_job, end_job = get_job_range(job_id)
    print(f'start_job = {start_job}, end_job = {end_job}')


start_job = 0, end_job = 5000


In [6]:
#JOB ARRAY SUBSETTING
A_g=A_g[:,start_job:end_job]
A_g_Processed=A_g_Processed[:,start_job:end_job]
A_c_Processed=A_c_Processed[:,start_job:end_job]
A_c=A_c[:,start_job:end_job]
X=X[:,start_job:end_job]
Y=Y[:,start_job:end_job]
Z=Z[:,start_job:end_job]

In [ ]:
##################################################

In [7]:
#constants
Cp=1004 #Jkg-1K-1
Cv=717 #Jkg-1K-1
Rd=Cp-Cv #Jkg-1K-1
eps=0.608

Lx=data['xf'][-1].item()-data['xf'][0].item() #x length (km)
Ly=data['yf'][-1].item()-data['yf'][0].item() #y length (km)
Np=len(parcel['xh']) #number of lagrangian parcles
dt=data['time'][1].item()/1e9 #sec
dx=(data['xf'][1].item()-data['xf'][0].item())*1e3 #meters
dy=(data['yf'][1].item()-data['yf'][0].item())*1e3 #meters
xs=data['xf'].values*1000
ys=data['yf'].values*1000
zs=data['zf'].values*1000

def zf(z):
    k=z #z is the # level of z
    out=data['zf'].values[k]*1000
    return out
# def rho(x,y,z,t):
#     p=data['prs'].isel(xh=x,yh=y,zh=z,time=t).item()
#     p0=101325 #Pa
#     theta=data['th'].isel(xh=x,yh=y,zh=z,time=t).item()
#     T=theta*(p/p0)**(Rd/Cp)
#     qv=data['qv'].isel(xh=x,yh=y,zh=z,time=t).item()
#     # Tv=T*(1+eps*qv)
#     Tv=T*(eps+qv)/(eps*(1+qv))
#     rho = p/(Rd*Tv)
#     out=rho
#     return out
rho_data=data['rho'].data
def rho(x,y,z,t):
    # out=data['rho'].isel(xh=x,yh=y,zh=z,time=t).item()
    out=rho_data[t,z,y,x]
    return out
def m(t):
    m=0
    #triple sum
    for k in range(len('zh')):
        for j in range(len('yh')):
            for i in range(len('xh')):
                rho_out=rho(i,j,k,t)
                m+=rho_out*(zf(k+1)-zf(k))
    #triple sum
    m=m*dx*dy/Np
    out=m
    return out

In [8]:
def ed3d(A,t,z,y,x,type):
    #Get Z Locations
    zs=Z[t,:]
    ys=Y[t,:]
    xs=X[t,:]
    
    #Essential A_t-A_(t-1)
    D=A[t,:]-A[t-1,:]
    
    #Essentially the I function
    zyx_ind=np.where((zs==z)&(ys==y)&(xs==x))
    A_z=D[zyx_ind]
    
    #Esentially the H function
    if type=='e':
        A_sum=np.sum(A_z[A_z>0]) #entrainment
    if type=='d':
        A_sum=-np.sum(A_z[A_z<0]) #detrainment

    #REMOVING THE FIRST TIMESTEP
    if t==0:
        A_sum=0

    # #CONSTANT (APPLIED AFTER CALCULATION)
    # ############
    # m_out=m(t)
    # dz=zf(z+1)-zf(z);
    # dy=1000;dx=1000
    # # dx=Lx; dy=Ly #TESTING
    # constant=(m_out/dx/dy/dz/dt) 
    # # constant=1
    # A_sum*=constant
    return A_sum

In [19]:
# %%time
# z=10;y=100;x=450
# A=A_c
# for t in np.arange(len(data['time'])):
#     #Get Z Locations
#     zs=Z[t,:]
#     ys=Y[t,:]
#     xs=X[t,:]
    
#     #Essential A_t-A_(t-1)
#     D=A[t,:]-A[t-1,:]
    
#     #Essentially the I function
#     zyx_ind=np.where((zs==z)&(ys==y)&(xs==x))
#     A_z=D[zyx_ind]
    
#     #Esentially the H function
#     if type=='e':
#         A_sum=np.sum(A_z[A_z>0]) #entrainment
#     if type=='d':
#         A_sum=-np.sum(A_z[A_z<0]) #detrainment


CPU times: user 491 ms, sys: 15.7 ms, total: 506 ms
Wall time: 587 ms


In [9]:
# #TESTING TESTING
# A=A_c;
# type='e'
# Nt=len(data['time'])
# Nz=len(data['zh'])
# Ny=len(data['yh'])
# Nx=len(data['xh'])
# testNumber=Nz*Ny*Nx

# #TESTING SPEED
# import random
# import time
# # Randomly choose 1000 grid points in the dimensions
# random_grid_points = [
#     (random.randint(0, Nt-1), random.randint(0, Nz-1), random.randint(0, Ny-1), random.randint(0, Nx-1))
#     for _ in range(testNumber)
# ]

# # Timer start
# start_time = time.time()

# # Run the ed3d function on the selected grid points
# for count,point in enumerate(random_grid_points):
#     if count % 1e5 == 0: print(count*100/len(random_grid_points))
#     t, z, y, x = point
#     ed3d(A, t, z, y, x, type)  # Assuming ed3d is properly defined

# # Timer end
# end_time = time.time()

# # Calculate total time taken
# total_time = end_time - start_time
# print(f"Total time for processing {testNumber} random grid points: {total_time} seconds")

# Volume=(Nt*Nz*Ny*Nx)
# out=total_time*(Volume/testNumber)/60/60 
# print(f'{out} hours') 
# #estimated 2.68 hours for Ny*Nx for 2500 parcels per job
# #estimated 5 hours for Nz*Ny*Nx for 2500 parcels per job
# #assuming: estimated 27 days for Nt*Nz*Ny*Nx for 2500 parcels per job


0.0
2.8722426470588234
5.744485294117647
8.616727941176471
11.488970588235293
14.361213235294118
17.233455882352942
20.105698529411764
22.977941176470587
25.850183823529413
28.722426470588236
31.594669117647058
34.466911764705884
37.3391544117647
40.21139705882353
43.083639705882355
45.955882352941174
48.828125
51.700367647058826
54.572610294117645
57.44485294117647
60.3170955882353
63.189338235294116
66.06158088235294
68.93382352941177
71.8060661764706
74.6783088235294
77.55055147058823
80.42279411764706
83.29503676470588
86.16727941176471
89.03952205882354
91.91176470588235
94.78400735294117
97.65625
Total time for processing 3481600 random grid points: 146.19936966896057 seconds
5.401254490547711 hours


In [ ]:
def call_variables(t): 
    if np.mod(t,25)==0: print(f'loading variables for time {t}')
    variable='winterp'; w_data=data[variable].isel(time=t).data 
    variable='qv'; qv_data=data[variable].isel(time=t).data # get qc data
    variable='qc'; qc_data=data[variable].isel(time=t).data # get qc data
    variable='qi'; qi_data=data[variable].isel(time=t).data # get qc data
    qc_plus_qi=qc_data+qi_data
    variable='th'; th_data=data[variable].isel(time=t).data # get qc data
    variable='buoyancy'; buoyancy_data=data[variable].isel(time=t).data # get qc data
    
    import h5py
    with h5py.File(dir + 'Variable_Calculation/' + 'theta_e'+f'_{res}_{Np_str}'+'.h5', 'r') as f:
        theta_e_data = f['theta_e'][t]

    return w_data,qv_data,qc_data,qi_data,qc_plus_qi,th_data,buoyancy_data,theta_e_data

In [33]:
#Eulerian General and Cloudy Updrafts
##############
def get_indices_g(w_thresh1,qcqi_thresh,type):
        
    where1g_all_t = []  # List to collect indices for each time step
    where1g_all_z = []
    where1g_all_y = []
    where1g_all_x = []
    
    Nt = len(data['time'])
    for t in range(Nt):  
        [w_data,qv_data,qc_data,qi_data,qc_plus_qi,th_data,buoyancy_data,theta_e_data] = call_variables(t)  # Load variables for time t

        if type=='e':
            where1g_t = np.where((w_data>=w_thresh1)&(qc_plus_qi<qcqi_thresh))
        if type=='d':
            where1g_t = np.where((w_data<w_thresh1)|(qc_plus_qi>=qcqi_thresh))
        
        if where1g_t[0].size > 0:  # Ensure non-empty results
            where1g_all_t.append(np.full_like(where1g_t[0], t))  # Create time index array
            where1g_all_z.append(where1g_t[0])  # z indices
            where1g_all_y.append(where1g_t[1])  # y indices
            where1g_all_x.append(where1g_t[2])  # x indices
    
    # Convert lists to numpy arrays and concatenate
    where1g = (
        np.concatenate(where1g_all_t),
        np.concatenate(where1g_all_z),
        np.concatenate(where1g_all_y),
        np.concatenate(where1g_all_x)
    )
    return where1g


#Eulerian General and Cloudy Updrafts
##############

def get_indices_c(w_thresh2,qcqi_thresh,type):
    
    where1c_all_t = []  # List to collect indices for each time step
    where1c_all_z = []
    where1c_all_y = []
    where1c_all_x = []
    
    Nt = len(data['time'])
    for t in range(Nt):  
        [w_data,qv_data,qc_data,qi_data,qc_plus_qi,th_data,buoyancy_data,theta_e_data] = call_variables(t)  # Load variables for time t

        if type=='e':
            where1c_t = np.where((w_data >= w_thresh2) & (qc_plus_qi >= qcqi_thresh))
        if type=='d':
            where1c_t = np.where((w_data < w_thresh2) | (qc_plus_qi < qcqi_thresh))
        
        if where1c_t[0].size > 0:  # Ensure non-empty results
            where1c_all_t.append(np.full_like(where1c_t[0], t))  # Create time index array
            where1c_all_z.append(where1c_t[0])  # z indices
            where1c_all_y.append(where1c_t[1])  # y indices
            where1c_all_x.append(where1c_t[2])  # x indices
    
    # Convert lists to numpy arrays and concatenate
    where1c = (
        np.concatenate(where1c_all_t),
        np.concatenate(where1c_all_z),
        np.concatenate(where1c_all_y),
        np.concatenate(where1c_all_x)
    )
    return where1c

#TESTING
# w_thresh1=0.1
# qcqi_thresh=1e-6
# indices=np.where((w_data>=w_thresh1)&(qc_plus_qi<qcqi_thresh)) #USE IF LOADING FULL VARIABLE
# one=indices
# indices = get_indices_g(w_thresh1,qcqi_thresh,type='e') #USE IF LOADING TIMESTEP BY TIMESTEP
# two=indices
# np.all((one[0]==two[0])==True)

In [34]:
%%time
PROCESSING=False

#TESTING TESTING TESTING
tlen=len(data['time'])
zlen=len(data['zh'])
ylen=len(data['yh'])
xlen=len(data['xh'])

profile_array_e_g=np.zeros((tlen,zlen,ylen,xlen))
profile_array_d_g=np.zeros((tlen,zlen,ylen,xlen))
profile_array_e_c=np.zeros((tlen,zlen,ylen,xlen))
profile_array_d_c=np.zeros((tlen,zlen,ylen,xlen))


w_thresh1=0.1
qcqi_thresh=1e-6
# indices=np.where((w_data>=w_thresh1)&(qc_plus_qi<qcqi_thresh)) #USE IF LOADING FULL VARIABLE
indices = get_indices_g(w_thresh1,qcqi_thresh,type='e') #USE IF LOADING TIMESTEP BY TIMESTEP

for count, (t, z, y, x) in enumerate(zip(*indices)):
    if count*100/len(indices[0])>=10:break #TESTING
    
    if np.mod(count,10000)==0: print(f'{count*100/len(indices[0]):.2f}%')

    if PROCESSING==False:
        A_sum_e_g=ed3d(A_g,t,z,y,x,type='e')
    elif PROCESSING==True:
        A_sum_e_g=ed3d(A_g_Processed,t,z,y,x,type='e') #PROCESSING
    
    profile_array_e_g[t,z,y,x]+=A_sum_e_g 

0.00%
0.02%
0.04%
0.05%
0.07%
0.09%
0.11%
0.12%
0.14%
0.16%
0.18%
0.19%
0.21%
0.23%
0.25%
0.26%
0.28%
0.30%
0.32%
0.33%
0.35%
0.37%
0.39%
0.40%
0.42%
0.44%
0.46%
0.47%
0.49%
0.51%
0.53%
0.55%
0.56%
0.58%
0.60%
0.62%
0.63%
0.65%
0.67%
0.69%
0.70%
0.72%
0.74%
0.76%
0.77%
0.79%
0.81%
0.83%
0.84%
0.86%
0.88%
0.90%
0.91%
0.93%
0.95%
0.97%
0.98%
1.00%
1.02%
1.04%
1.06%
1.07%
1.09%
1.11%
1.13%
1.14%
1.16%
1.18%
1.20%
1.21%
1.23%
1.25%
1.27%
1.28%
1.30%
1.32%
1.34%
1.35%
1.37%
1.39%
1.41%
1.42%
1.44%
1.46%
1.48%
1.49%
1.51%
1.53%
1.55%
1.57%
1.58%
1.60%
1.62%
1.64%
1.65%
1.67%
1.69%
1.71%
1.72%
1.74%
1.76%
1.78%
1.79%
1.81%
1.83%
1.85%
1.86%
1.88%
1.90%
1.92%
1.93%
1.95%
1.97%
1.99%
2.00%
2.02%
2.04%
2.06%
2.08%
2.09%
2.11%
2.13%
2.15%
2.16%
2.18%
2.20%
2.22%
2.23%
2.25%
2.27%
2.29%
2.30%
2.32%
2.34%
2.36%
2.37%
2.39%
2.41%
2.43%
2.44%
2.46%
2.48%
2.50%
2.51%
2.53%
2.55%
2.57%
2.59%
2.60%
2.62%
2.64%
2.66%
2.67%
2.69%
2.71%
2.73%
2.74%
2.76%
2.78%
2.80%
2.81%
2.83%
2.85%
2.87%
2.88%
2.90%
2.92

In [38]:
#Total about 24 minutes per job for 200 jobs ((120+26)*10)/60 
#Total about 20.67 minutes per job for 400 jobs ((120+4)*10)/60 

In [ ]:
########################################################################################

In [ ]:
#TURN PROCESSING ON OR OFF
PROCESSING=False
PROCESSING=True

#SET UP TO RUN WITH JOB_ARRAY
# calc_entrain=False #*** 
# calc_detrain=False #***
calc_entrain=True #*** 
calc_detrain=True #***

#creates 2d storage array

tlen=len(data['time'])
zlen=len(data['zh'])
ylen=len(data['yh'])
xlen=len(data['xh'])

profile_array_e_g=np.zeros((tlen,zlen,ylen,xlen))
profile_array_d_g=np.zeros((tlen,zlen,ylen,xlen))
profile_array_e_c=np.zeros((tlen,zlen,ylen,xlen))
profile_array_d_c=np.zeros((tlen,zlen,ylen,xlen))
    
#Adding to Profile Array
# import itertools
# ts = range(0, 141)  # ts from 0 to 140
# zs = range(0, 34)   # zs from 0 to 34
# for count, (t, z) in enumerate(itertools.product(ts, zs)):

#GENERAL UPDRAFTS

#ENTRAINMENT
if calc_entrain==True:
    w_thresh1=0.1
    qcqi_thresh=1e-6
    # indices=np.where((w_data>=w_thresh1)&(qc_plus_qi<qcqi_thresh)) #USE IF LOADING FULL VARIABLE
    indices = get_indices_g(w_thresh1,qcqi_thresh,type='e') #USE IF LOADING TIMESTEP BY TIMESTEP
    
    for count, (t, z, y, x) in enumerate(zip(*indices)):
        if np.mod(count,10000)==0: print(f'{count*100/len(indices[0]):.2f}%')

        if PROCESSING==False:
            A_sum_e_g=ed3d(A_g,t,z,y,x,type='e')
        elif PROCESSING==True:
            A_sum_e_g=ed3d(A_g_Processed,t,z,y,x,type='e') #PROCESSING
        
        profile_array_e_g[t,z,y,x]+=A_sum_e_g 

#DETRAINMENT
if calc_detrain==True:
    w_thresh1=0.1
    qcqi_thresh=1e-6
    # indices=np.where((w_data<w_thresh1)| (qc_plus_qi>=qcqi_thresh)) #NEGATION OF THRESHOLD
    indices = get_indices_g(w_thresh1,qcqi_thresh,type='d') #USE IF LOADING TIMESTEP BY TIMESTEP

    for count, (t, z, y, x) in enumerate(zip(*indices)):
        if np.mod(count,10000)==0: print(f'{count*100/len(indices_[0]):.2f}%')

        if PROCESSING==False:
            A_sum_d_g=ed3d(A_g,t,z,y,x,type='d')
        elif PROCESSING==True:
            A_sum_d_g=ed3d(A_g_Processed,t,z,y,x,type='d') #PROCESSING
        
        profile_array_d_g[t,z,y,x]+=A_sum_d_g 


#CLOUDY UPDRAFTS

#ENTRAINMENT
if calc_entrain==True:
    w_thresh2=0.5
    qcqi_thresh=1e-6
    # indices=np.where((w_data>=w_thresh2)&(qc_plus_qi>=qcqi_thresh)) #USE IF LOADING FULL VARIABLE
    indices=get_indices_c(w_thresh2,qcqi_thresh,type='e') 

    for count, (t, z, y, x) in enumerate(zip(*indices)):
        if np.mod(count,10000)==0: print(f'{count*100/len(indices[0]):.2f}%')

        if PROCESSING==False:
            A_sum_e_c=ed3d(A_c,t,z,y,x,type='e') 
        elif PROCESSING==True:
            A_sum_e_c=ed3d(A_c_Processed,t,z,y,x,type='e') #PROCESSING

        profile_array_e_c[t,z,y,x]+=A_sum_e_c


#DETRAINMENT
if calc_detrain==True:
    w_thresh2=0.5
    qcqi_thresh=1e-6
    # indices=np.where((w_data<w_thresh2)|(qc_plus_qi<qcqi_thresh))
    indices=get_indices_c(w_thresh2,qcqi_thresh,type='d') 
    
    for count, (t, z, y, x) in enumerate(zip(*indices)):
        if np.mod(count,10000)==0: print(f'{count*100/len(indices[0]):.2f}%')

        if PROCESSING==False:
            A_sum_d_c=ed3d(A_c,t,z,y,x,type='d')
        elif PROCESSING==True:
            A_sum_d_c=ed3d(A_c_Processed,t,z,y,x,type='d')
        
        profile_array_d_c[t,z,y,x]+=A_sum_d_c

#SAVING
if calc_entrain==True:
    dir3=dir+f'Project_Algorithms/Entrainment/job_out/3D_entrainment_profiles{job_id}_2.h5'
    with h5py.File(dir3, "w") as h5f:
        h5f.create_dataset("profile_array_e_g", data=profile_array_e_g)
        h5f.create_dataset("profile_array_e_c", data=profile_array_e_c)
if calc_detrain==True:
    dir3=dir+f'Project_Algorithms/Entrainment/job_out/3D_detrainment_profiles{job_id}_2.h5'
    with h5py.File(dir3, "w") as h5f:
        h5f.create_dataset("profile_array_d_g", data=profile_array_d_g)
        h5f.create_dataset("profile_array_d_c", data=profile_array_d_c)

In [ ]:
################################################################

In [ ]:
#RECOMBING DATA AFTERWARDS
#######################################################
#COMBINING JOB_ARRAYS (RUN AFTER ALL JOB_ARRAYS ARE FINISHED)

tlen=len(netCDF['time'])
zlen=len(netCDF['zh'])
ylen=len(netCDF['yh'])
xlen=len(netCDF['xh'])
profile_array_e_g=np.zeros((tlen,zlen,ylen,xlen))
profile_array_d_g=np.zeros((tlen,zlen,ylen,xlen))
profile_array_e_c=np.zeros((tlen,zlen,ylen,xlen))
profile_array_d_c=np.zeros((tlen,zlen,ylen,xlen))

#ENTRAINMENT
tlen1=0
for job_id in np.arange(1,60+1):
    dir3=dir+f'Project_Algorithms/Entrainment/job_out/3D_entrainment_profiles{job_id}_2.h5'
    with h5py.File(dir3, "r") as h5f:
        for t in np.arange(len('time')):
            for z in np.arange(len('zh')):
                profile_array_e_g[t,z] = h5f["profile_array_e_g"][t,z]
                profile_array_e_c[t,z] = h5f["profile_array_e_c"][t,z]

#SAVING
dir3=dir+f'Project_Algorithms/Entrainment/3D_entrainment_profiles_2.h5'
with h5py.File(dir3, "w") as h5f:
    h5f.create_dataset("profile_array_e_g", data=profile_array_e_g)
    h5f.create_dataset("profile_array_e_c", data=profile_array_e_c)


#ENTRAINMENT
tlen1=0
for job_id in np.arange(1,60+1):
    dir3=dir+f'Project_Algorithms/Entrainment/job_out/3D_detrainment_profiles{job_id}_2.h5'
    with h5py.File(dir3, "r") as h5f:
        for t in np.arange(len('time')):
            for z in np.arange(len('zh')):
                profile_array_d_g[t,z] = h5f["profile_array_d_g"][t,z]
                profile_array_d_c[t,z] = h5f["profile_array_d_c"][t,z]

#SAVING
dir3=dir+f'Project_Algorithms/Entrainment/3D_detrainment_profiles_2.h5'
with h5py.File(dir3, "w") as h5f:
    h5f.create_dataset("profile_array_d_g", data=profile_array_d_g)
    h5f.create_dataset("profile_array_d_c", data=profile_array_d_c)


In [ ]:
##########################################################################

In [ ]:
#######################################################
#READING BACK IN 
dir3=dir+'Project_Algorithms/Entrainment/3D_entrainment_profiles_2.h5'
with h5py.File(dir3, "r") as h5f:
    profile_array_e_g = h5f["profile_array_e_g"][:]
    profile_array_e_c = h5f["profile_array_e_c"][:]
dir3=dir+'Project_Algorithms/Entrainment/3D_detrainment_profiles_2.h5'
with h5py.File(dir3, "r") as h5f:
    profile_array_d_g = h5f["profile_array_d_g"][:]
    profile_array_d_c = h5f["profile_array_d_c"][:]

In [ ]:
#DOMAIN SUBSETTING
ocean_percent=2/8

left_to_coast=data['xh'][0]+(data['xh'][-1]-data['xh'][0])*ocean_percent
where_coast_xh=np.where(data['xh']>=left_to_coast)[0][0]#-25
where_coast_xf=np.where(data['xf']>=left_to_coast)[0][0]#-25
end_xh=len(data['xh'])-1-50
end_xf=len(data['xf'])-1-50

print(f'x in {0}:{where_coast_xh-1} FOR SEA')
print(f'x in {where_coast_xh}:{end_xh} FOR LAND')
# t_end=78 
# if res=='250m':t_end=410
# print(f't in {0}:{t_end} (6.5 hours)')
t_start=36 
print(f't in {t_start}:end (8 hours)')

profile_array_e_g=profile_array_e_g[slice(0,78+1),:,:,slice(where_coast_xh,end_xh+1)]
profile_array_d_g=profile_array_d_g[slice(0,78+1),:,:,slice(where_coast_xh,end_xh+1)]
profile_array_e_c=profile_array_e_c[slice(0,78+1),:,:,slice(where_coast_xh,end_xh+1)]
profile_array_d_c=profile_array_d_c[slice(0,78+1),:,:,slice(where_coast_xh,end_xh+1)]

In [ ]:
def apply_constant(profile_array,apply):
    if apply==True:
        Nt=profile_array.shape[0]
        Nz=profile_array.shape[1]
    
        profile_array/=(dx*dy*dt)
        for t in np.arange(Nt):
            profile_array[t]*=m(t)
        for z in np.arange(Nz):
            dz=zf(z+1)-zf(z)
            profile_array[:,z]/=dz
    return profile_array

In [ ]:
from matplotlib.gridspec import GridSpec
fig = plt.figure(figsize=(10, 8))
gs = GridSpec(2, 2, figure=fig)

######
cmap1 = plt.cm.viridis
cmap2 = plt.cm.seismic 
n_levels=29
######

type='general'
# type='cloudy'

if type=="general":
    profile_array_e=profile_array_e_g
    profile_array_d=profile_array_d_g
if type=="cloudy":
    profile_array_e=profile_array_e_c
    profile_array_d=profile_array_d_c

#APPLY CONSTANTS TO ENTRAINMENT VALUE
##################################################
profile_array_e=apply_constant(profile_array_e,True)
# profile_array_e=apply_constant(profile_array_e,False) #TESTING
profile_array_d=apply_constant(profile_array_d,True)
# profile_array_d=apply_constant(profile_array_d,False) #TESTING
##################################################


ax1 = fig.add_subplot(gs[0, 0])
out = np.mean(profile_array_e, axis=(2, 3))
cf1 = ax1.contourf(out.T, cmap=cmap1)
fig.colorbar(cf1, ax=ax1, orientation='vertical')
ax1.set_title('3D Entrainment Horizontal Mean (E)')
ax1.set_xlabel('t')
ax1.set_ylabel('z')

ax2 = fig.add_subplot(gs[0, 1])
out = np.mean(profile_array_d, axis=(2, 3))
cf2 = ax2.contourf(out.T, cmap=cmap1)
fig.colorbar(cf2, ax=ax2, orientation='vertical')
ax2.set_title('3D Entrainment Horizontal Mean (D)')
ax2.set_xlabel('t')
ax2.set_ylabel('z')


ax3 = fig.add_subplot(gs[1, 0])
profile_array_net = np.mean(profile_array_e-profile_array_d, axis=(2, 3))

# # Normalize with a balanced vmin and vmax
# profile_array_net=out
# vmin=-np.max(abs(profile_array_net)); vmax=+np.max(abs(profile_array_net))
# levels = np.linspace(vmin, vmax, n_levels)
# norm = mcolors.BoundaryNorm(boundaries=levels, ncolors=256)
# cf3 = ax3.contourf(out.T, cmap=cmap2,norm=norm,levels=levels)

vmin=-np.max(abs(profile_array_net)); vmax=+np.max(abs(profile_array_net))
levels = np.linspace(vmin, vmax, n_levels)

cf3 = ax3.contourf(profile_array_net.T, cmap=cmap2,vmin=vmin,vmax=vmax,levels=levels)
fig.colorbar(cf3, ax=ax3, orientation='vertical')
ax3.set_title('3D Entrainment Horizontal Mean (D)')
ax3.set_xlabel('t')
ax3.set_ylabel('z')

plt.tight_layout()

In [ ]:
fig = plt.figure(figsize=(10, 8))
gs = GridSpec(2, 2, figure=fig)

######
cmap1 = plt.cm.viridis
cmap2 = plt.cm.seismic 
n_levels=29
######

ax1 = fig.add_subplot(gs[0, 0])
out = np.mean(profile_array_e, axis=(0, 2))
cf1 = ax1.contourf(out, levels=50, cmap=cmap1)
fig.colorbar(cf1, ax=ax1, orientation='vertical')
ax1.set_title('3D Entrainment x-z Mean (e)')
ax1.set_xlabel('x')
ax1.set_ylabel('z')

ax2 = fig.add_subplot(gs[0, 1])
out = np.mean(profile_array_d, axis=(0, 2))
cf2 = ax2.contourf(out, levels=50, cmap=cmap1)
fig.colorbar(cf2, ax=ax2, orientation='vertical')
ax2.set_title('3D Entrainment x-z Mean (d)')
ax2.set_xlabel('x')
ax2.set_ylabel('z')


profile_array_net=np.mean(profile_array_e-profile_array_d,axis=(0,2))
nlevel = 29
vmin=-np.max(abs(profile_array_net)); vmax=+np.max(abs(profile_array_net))
levels = np.linspace(vmin, vmax, nlevel)

ax3 = fig.add_subplot(gs[1, 0])
cf3 = ax3.contourf(profile_array_net, levels=levels, vmin=vmin,vmax=vmax,cmap=cmap2)
fig.colorbar(cf3, ax=ax3, orientation='vertical')
ax3.set_title('3D Entrainment x-z Mean (d)')
ax3.set_xlabel('x')
ax3.set_ylabel('z')

plt.tight_layout()

# for axis in [ax1,ax2,ax3]:
#     axis.set_xlim(left=-where_coast_xh)

In [ ]:
correction=1000**2
correction=1

plt.plot(np.mean(profile_array_e,axis=(0,2,3))*correction,data['zh'],label='entrainment')
plt.plot(np.mean(profile_array_d,axis=(0,2,3))*correction,data['zh'],label='detrainment')
plt.plot(np.mean(profile_array_e-profile_array_d,axis=(0,2,3))*correction,data['zh'],label='entrainment - detrainment')

plt.axvline(0,color='k',linestyle='dashed')

plt.legend(); plt.title('3D Entrainment and Detrainment Using Lagrangian Binary Array')

from matplotlib.ticker import ScalarFormatter
formatter = ScalarFormatter(useMathText=True)
formatter.set_scientific(True)
formatter.set_powerlimits((-1, 1))
plt.gca().xaxis.set_major_formatter(formatter)